## Query 1: Vehículos con más kilómetros recorridos

```sql
SELECT 
    v.vehicle_id,
    v.license_plate,
    v.vehicle_type,
    SUM(r.distance_km) AS total_km_recorridos
FROM trips t
JOIN vehicles v ON t.vehicle_id = v.vehicle_id
JOIN routes r ON t.route_id = r.route_id
GROUP BY v.vehicle_id, v.license_plate, v.vehicle_type
ORDER BY total_km_recorridos DESC
LIMIT 10;
´

Objetivo
Identificar cuáles son los 10 vehículos que más kilómetros recorrieron dentro de la operación registrada.

Explicación
La consulta relaciona la tabla de viajes con la de vehículos y rutas. Luego suma la distancia de cada ruta recorrida por vehículo para obtener el total de kilómetros acumulados.

Problema de negocio que resuelve
Esta query permite reconocer cuáles son las unidades más utilizadas de la flota. Esta información es importante para el seguimiento operativo, la planificación del mantenimiento y la detección de vehículos con mayor nivel de desgaste.
El resultado muestra un ranking de los 10 vehículos con mayor cantidad de kilómetros recorridos, junto con su patente, tipo y total acumulado.


## Query 2: Conductores con más kilómetros recorridos

```sql
SELECT 
    d.driver_id,
    d.first_name,
    d.last_name,
    SUM(r.distance_km) AS total_km_recorridos
FROM trips t
JOIN drivers d ON t.driver_id = d.driver_id
JOIN routes r ON t.route_id = r.route_id
GROUP BY d.driver_id, d.first_name, d.last_name
ORDER BY total_km_recorridos DESC
LIMIT 10;

Objetivo
Identificar cuáles son los 10 conductores que más kilómetros recorrieron en los viajes registrados.

La consulta relaciona la tabla trips con drivers y routes. Luego suma la distancia de las rutas asociadas a cada viaje para calcular el total de kilómetros recorridos por cada conductor.

Dato de Negocio:
Esta query permite analizar la carga operativa de los conductores y detectar cuáles son los que tienen mayor participación en la operación logística. Esta información puede ser útil para distribución de trabajo, seguimiento del desempeño y control operativo.

El resultado muestra un ranking de los 10 conductores con mayor cantidad de kilómetros recorridos.


## Query 3: Vehículos con mayor costo de mantenimiento

```sql
SELECT 
    v.vehicle_id,
    v.license_plate,
    v.vehicle_type,
    SUM(m.cost) AS costo_total_mantenimiento
FROM maintenance m
JOIN vehicles v ON m.vehicle_id = v.vehicle_id
GROUP BY v.vehicle_id, v.license_plate, v.vehicle_type
ORDER BY costo_total_mantenimiento DESC
LIMIT 10;

Objetivo
Identificar cuáles son los 10 vehículos que acumularon mayor costo de mantenimiento.

Explicación
La consulta relaciona la tabla maintenance con la tabla vehicles a partir del identificador del vehículo. Luego suma los costos de mantenimiento registrados para cada unidad y ordena los resultados de mayor a menor.

Problema de negocio que resuelve
Esta query permite detectar qué vehículos generan mayores gastos de mantenimiento. Esta información es útil para evaluar el desgaste de la flota, controlar costos operativos y apoyar decisiones sobre reparación, seguimiento o reemplazo de unidades.

Resultado esperado
El resultado muestra un ranking de los 10 vehículos con mayor costo acumulado de mantenimiento.


## Query Intermedia 1: Promedio de entregas por conductor en los últimos 6 meses

```sql
SELECT 
    d.driver_id,
    d.first_name || ' ' || d.last_name AS conductor,
    COUNT(del.delivery_id) AS total_entregas,
    COUNT(DISTINCT t.trip_id) AS total_viajes,
    ROUND(COUNT(del.delivery_id)::numeric / NULLIF(COUNT(DISTINCT t.trip_id), 0), 2) AS promedio_entregas_por_viaje
FROM drivers d
JOIN trips t ON d.driver_id = t.driver_id
JOIN deliveries del ON t.trip_id = del.trip_id
WHERE t.departure_datetime >= CURRENT_DATE - INTERVAL '6 months'
GROUP BY d.driver_id, d.first_name, d.last_name
HAVING COUNT(del.delivery_id) > 0
ORDER BY promedio_entregas_por_viaje DESC;

Objetivo
Analizar la cantidad de entregas realizadas por cada conductor durante los últimos 6 meses y calcular su promedio de entregas por viaje.

Explicación
La consulta relaciona las tablas drivers, trips y deliveries para obtener información conjunta sobre conductores, viajes y entregas. Luego filtra únicamente los viajes realizados en los últimos 6 meses, cuenta la cantidad total de entregas y la cantidad de viajes por conductor, y finalmente calcula el promedio de entregas por viaje.

Problema de negocio que resuelve
Esta query permite evaluar el nivel de actividad reciente de los conductores y detectar diferencias en la carga operativa. Puede ser útil para analizar desempeño, distribución del trabajo y productividad dentro de la operación logística.

Resultado esperado
El resultado muestra cada conductor con su total de entregas, total de viajes y el promedio de entregas por viaje en los últimos 6 meses, ordenado de mayor a menor.


## Query Intermedia 2: Rutas con mayor cantidad de viajes completados

```sql
SELECT 
    r.route_id,
    r.origin_city,
    r.destination_city,
    COUNT(t.trip_id) AS total_viajes,
    AVG(t.fuel_consumed_liters) AS promedio_combustible,
    AVG(t.total_weight_kg) AS promedio_carga
FROM routes r
JOIN trips t ON r.route_id = t.route_id
WHERE t.status = 'completed'
GROUP BY r.route_id, r.origin_city, r.destination_city
HAVING COUNT(t.trip_id) >= 10
ORDER BY total_viajes DESC;

Objetivo
Identificar cuáles son las rutas con mayor cantidad de viajes completados y obtener métricas promedio de operación sobre ellas.

Explicación
La consulta relaciona las tablas routes y trips, tomando únicamente los viajes con estado completed. Luego agrupa la información por ruta y calcula la cantidad total de viajes, el consumo promedio de combustible y la carga promedio transportada. Además, se filtran solo las rutas con al menos 10 viajes completados.

Resuelve:
Esta query permite reconocer cuáles son las rutas más utilizadas dentro de la operación y analizar algunas métricas clave sobre ellas. Esto puede ser útil para identificar trayectos de alta demanda, estudiar costos operativos y apoyar decisiones de planificación logística.

El resultado muestra las rutas con más viajes completados, junto con su cantidad de viajes, consumo promedio de combustible y carga promedio transportada.


## Query Intermedia 3: Rutas con mayor cantidad de entregas demoradas

```sql
SELECT 
    r.route_id,
    r.origin_city,
    r.destination_city,
    COUNT(del.delivery_id) AS total_entregas,
    SUM(CASE 
        WHEN del.delivered_datetime > del.scheduled_datetime THEN 1
        ELSE 0
    END) AS entregas_demoradas
FROM routes r
JOIN trips t ON r.route_id = t.route_id
JOIN deliveries del ON t.trip_id = del.trip_id
WHERE del.delivered_datetime IS NOT NULL
GROUP BY r.route_id, r.origin_city, r.destination_city
HAVING COUNT(del.delivery_id) >= 20
ORDER BY entregas_demoradas DESC;

Objetivo
Identificar cuáles son las rutas que concentran mayor cantidad de entregas realizadas con demora.

Explicación
La consulta relaciona routes, trips y deliveries para analizar las entregas realizadas en cada ruta. Luego compara la fecha programada con la fecha real de entrega y cuenta cuántas entregas fueron demoradas. Además, se filtran solo las rutas con un volumen mínimo de entregas para que el análisis sea más representativo.

Problema de negocio que resuelve
Esta query permite detectar rutas con problemas de puntualidad, lo que puede ayudar a revisar tiempos estimados, planificación logística y desempeño operativo en trayectos específicos.

Resultado esperado
El resultado muestra las rutas con mayor cantidad de entregas demoradas, junto con el total de entregas registradas en cada una.

## Query Intermedia 4: Cantidad de viajes completados por mes

```sql
SELECT 
    DATE_TRUNC('month', departure_datetime) AS mes,
    COUNT(*) AS total_viajes
FROM trips
WHERE status = 'completed'
GROUP BY DATE_TRUNC('month', departure_datetime)
ORDER BY mes;

Objetivo
Analizar cómo se distribuyen los viajes completados a lo largo del tiempo, agrupándolos por mes.

Explicación
La consulta toma los viajes con estado completed y utiliza la función DATE_TRUNC('month', departure_datetime) para agruparlos por mes. Luego cuenta cuántos viajes hubo en cada período.

Problema de negocio que resuelve
Esta query permite observar la evolución mensual de la operación logística. Puede ser útil para detectar períodos de mayor o menor actividad, analizar tendencias y apoyar decisiones de planificación.

Observación del resultado
A partir del resultado se observa una distribución mensual bastante estable en la cantidad de viajes completados. En la mayoría de los meses, el total se mantiene cercano a los 4.300 o 4.400 viajes, lo que sugiere un volumen operativo relativamente constante.

También se aprecia que marzo de 2024 presenta un valor menor que el resto, con 2.155 viajes, posiblemente porque corresponde al inicio del período analizado o a un mes con menor cantidad de registros disponibles.


## Query Intermedia 5: Ciudades destino con mayor peso promedio entregado

```sql
SELECT 
    r.destination_city,
    COUNT(del.delivery_id) AS total_entregas,
    AVG(del.package_weight_kg) AS peso_promedio_paquetes
FROM deliveries del
JOIN trips t ON del.trip_id = t.trip_id
JOIN routes r ON t.route_id = r.route_id
GROUP BY r.destination_city
HAVING COUNT(del.delivery_id) >= 50
ORDER BY peso_promedio_paquetes DESC;


Objetivo
Analizar qué ciudades destino reciben, en promedio, paquetes de mayor peso.

Explicación
La consulta relaciona las tablas deliveries, trips y routes para asociar cada entrega con su ciudad de destino. Luego agrupa los resultados por ciudad, cuenta la cantidad de entregas y calcula el peso promedio de los paquetes entregados. Se filtran únicamente las ciudades con al menos 50 entregas registradas para asegurar que el análisis sea representativo.

Problema de negocio que resuelve
Esta query permite identificar destinos donde la operación logística maneja paquetes más pesados en promedio. Esta información puede ser útil para planificar recursos, tipos de vehículo y organización de entregas según la demanda de cada ciudad.

Resultado esperado
El resultado muestra las ciudades destino con su cantidad total de entregas y el peso promedio de los paquetes recibidos, ordenadas de mayor a menor.


## Query Compleja 1: Ranking de eficiencia de rutas considerando tiempo, combustible y entregas exitosas

```sql
WITH route_metrics AS (
    SELECT 
        r.route_id,
        r.origin_city,
        r.destination_city,
        COUNT(DISTINCT t.trip_id) AS total_viajes,
        AVG(EXTRACT(EPOCH FROM (t.arrival_datetime - t.departure_datetime)) / 3600) AS promedio_horas,
        AVG(t.fuel_consumed_liters) AS promedio_combustible,
        COUNT(del.delivery_id) AS total_entregas,
        SUM(CASE 
            WHEN del.delivery_status = 'delivered' THEN 1
            ELSE 0
        END) AS entregas_exitosas
    FROM routes r
    JOIN trips t ON r.route_id = t.route_id
    LEFT JOIN deliveries del ON t.trip_id = del.trip_id
    WHERE t.status = 'completed'
      AND t.arrival_datetime IS NOT NULL
    GROUP BY r.route_id, r.origin_city, r.destination_city
),
route_efficiency AS (
    SELECT 
        route_id,
        origin_city,
        destination_city,
        total_viajes,
        ROUND(promedio_horas::numeric, 2) AS promedio_horas,
        ROUND(promedio_combustible::numeric, 2) AS promedio_combustible,
        total_entregas,
        entregas_exitosas,
        ROUND((entregas_exitosas * 100.0 / NULLIF(total_entregas, 0))::numeric, 2) AS porcentaje_exito
    FROM route_metrics
)
SELECT 
    route_id,
    origin_city,
    destination_city,
    total_viajes,
    promedio_horas,
    promedio_combustible,
    total_entregas,
    entregas_exitosas,
    porcentaje_exito,
    RANK() OVER (
        ORDER BY porcentaje_exito DESC, promedio_horas ASC, promedio_combustible ASC
    ) AS ranking_eficiencia
FROM route_efficiency
ORDER BY ranking_eficiencia;

Objetivo
Evaluar la eficiencia de las rutas combinando distintas métricas operativas, como tiempo promedio de viaje, consumo promedio de combustible y porcentaje de entregas exitosas.

Explicación
La consulta utiliza dos CTEs. En el primero se calculan métricas generales por ruta a partir de los viajes y entregas asociadas. En el segundo se prepara el porcentaje de éxito de las entregas. Finalmente, se aplica una función de ventana RANK() para construir un ranking de eficiencia, priorizando las rutas con mayor porcentaje de entregas exitosas, menor tiempo promedio y menor consumo promedio de combustible.

Problema de negocio que resuelve
Esta query permite comparar el desempeño de las rutas desde una mirada más integral, no solo considerando volumen de viajes, sino también calidad de servicio y eficiencia operativa. Puede servir para detectar las rutas más convenientes y aquellas que requieren revisión o mejora.

Resultado esperado
Ranking de rutas ordenadas..



## Query Compleja 2: Evolución semestral de viajes entre ciudades con comparación contra el semestre anterior

```sql
WITH viajes_por_semestre AS (
    SELECT 
        r.origin_city,
        r.destination_city,
        DATE_TRUNC('year', t.departure_datetime) +
        CASE
            WHEN EXTRACT(MONTH FROM t.departure_datetime) <= 6
                THEN INTERVAL '0 months'
            ELSE INTERVAL '6 months'
        END AS semestre,
        COUNT(t.trip_id) AS total_viajes
    FROM trips t
    JOIN routes r ON t.route_id = r.route_id
    WHERE t.status = 'completed'
    GROUP BY 
        r.origin_city,
        r.destination_city,
        DATE_TRUNC('year', t.departure_datetime) +
        CASE
            WHEN EXTRACT(MONTH FROM t.departure_datetime) <= 6
                THEN INTERVAL '0 months'
            ELSE INTERVAL '6 months'
        END
),
comparacion_semestral AS (
    SELECT 
        origin_city,
        destination_city,
        semestre,
        total_viajes,
        LAG(total_viajes) OVER (
            PARTITION BY origin_city, destination_city
            ORDER BY semestre
        ) AS viajes_semestre_anterior
    FROM viajes_por_semestre
)
SELECT 
    origin_city,
    destination_city,
    semestre,
    total_viajes,
    viajes_semestre_anterior,
    total_viajes - viajes_semestre_anterior AS variacion_absoluta,
    ROUND(
        ((total_viajes - viajes_semestre_anterior) * 100.0 / NULLIF(viajes_semestre_anterior, 0))::numeric,
        2
    ) AS variacion_porcentual
FROM comparacion_semestral
ORDER BY origin_city, destination_city, semestre;


Objetivo
Analizar la evolución semestral de la cantidad de viajes realizados entre ciudades y comparar cada semestre con el anterior.

Explicación
La consulta agrupa los viajes completados por ciudad de origen, ciudad de destino y semestre. Luego utiliza la función LAG() para recuperar el total de viajes del semestre anterior dentro de cada conexión entre ciudades. Finalmente, calcula la variación absoluta y porcentual entre ambos períodos.

Problema de negocio que resuelve
Esta query permite observar cómo evoluciona la demanda de transporte entre distintas ciudades a lo largo del tiempo. De esta manera, ayuda a identificar conexiones con crecimiento, caída o estabilidad en la operación logística.

Resultado esperado
El resultado muestra, para cada par origen-destino y semestre, el total de viajes realizados y su comparación con el semestre anterior.




## Query Compleja 3: Ranking de ciudades destino según entregas exitosas y participación sobre el total

```sql
WITH entregas_por_ciudad AS (
    SELECT 
        r.destination_city,
        COUNT(del.delivery_id) AS total_entregas,
        SUM(CASE 
            WHEN del.delivery_status = 'delivered' THEN 1
            ELSE 0
        END) AS entregas_exitosas
    FROM deliveries del
    JOIN trips t ON del.trip_id = t.trip_id
    JOIN routes r ON t.route_id = r.route_id
    GROUP BY r.destination_city
),
participacion_ciudad AS (
    SELECT 
        destination_city,
        total_entregas,
        entregas_exitosas,
        ROUND((entregas_exitosas * 100.0 / NULLIF(total_entregas, 0))::numeric, 2) AS porcentaje_exito,
        ROUND((total_entregas * 100.0 / SUM(total_entregas) OVER ())::numeric, 2) AS participacion_total
    FROM entregas_por_ciudad
)
SELECT 
    destination_city,
    total_entregas,
    entregas_exitosas,
    porcentaje_exito,
    participacion_total,
    RANK() OVER (ORDER BY entregas_exitosas DESC) AS ranking_ciudad
FROM participacion_ciudad
ORDER BY ranking_ciudad;


Objetivo
Identificar qué ciudades destino concentran mayor cantidad de entregas exitosas y cuál es su peso relativo dentro del total de entregas del sistema.

Explicación
La consulta primero agrupa las entregas por ciudad destino y calcula tanto el total de entregas como las entregas exitosas. Luego obtiene dos indicadores: el porcentaje de éxito dentro de cada ciudad y la participación de esa ciudad sobre el total general de entregas. Finalmente, aplica una función de ventana RANK() para ordenar las ciudades según la cantidad de entregas exitosas.

Problema de negocio que resuelve
Esta query permite detectar cuáles son los destinos más relevantes para la operación logística y qué tan bien se está cumpliendo el servicio en cada uno. Esto puede ser útil para priorizar recursos, evaluar desempeño comercial y enfocar mejoras en destinos estratégicos.

Resultado esperado
El resultado muestra un ranking resumido de ciudades destino con su volumen total de entregas, cantidad de entregas exitosas, porcentaje de éxito y participación dentro de la operación total.


## Query Compleja 4: Ranking de tipos de vehículo según uso operativo y costo de mantenimiento

```sql
WITH metricas_por_vehiculo AS (
    SELECT 
        v.vehicle_id,
        v.vehicle_type,
        COUNT(DISTINCT t.trip_id) AS total_viajes,
        COALESCE(SUM(r.distance_km), 0) AS km_totales,
        COALESCE(SUM(m.cost), 0) AS costo_mantenimiento_total
    FROM vehicles v
    LEFT JOIN trips t ON v.vehicle_id = t.vehicle_id AND t.status = 'completed'
    LEFT JOIN routes r ON t.route_id = r.route_id
    LEFT JOIN maintenance m ON v.vehicle_id = m.vehicle_id
    GROUP BY v.vehicle_id, v.vehicle_type
),
resumen_por_tipo AS (
    SELECT 
        vehicle_type,
        COUNT(vehicle_id) AS cantidad_vehiculos,
        SUM(total_viajes) AS viajes_totales,
        SUM(km_totales) AS km_totales,
        SUM(costo_mantenimiento_total) AS costo_total_mantenimiento,
        ROUND(
            (SUM(costo_mantenimiento_total) / NULLIF(SUM(km_totales), 0))::numeric,
            2
        ) AS costo_por_km
    FROM metricas_por_vehiculo
    GROUP BY vehicle_type
)
SELECT 
    vehicle_type,
    cantidad_vehiculos,
    viajes_totales,
    km_totales,
    costo_total_mantenimiento,
    costo_por_km,
    RANK() OVER (ORDER BY costo_por_km ASC) AS ranking_eficiencia_costo
FROM resumen_por_tipo
ORDER BY ranking_eficiencia_costo;



Objetivo
Comparar los distintos tipos de vehículo según su nivel de uso operativo y su costo de mantenimiento relativo a los kilómetros recorridos.

Explicación
La consulta primero calcula métricas por vehículo, como cantidad de viajes, kilómetros recorridos y costo total de mantenimiento. Luego agrupa esa información por tipo de vehículo para obtener un resumen general por categoría. Finalmente, aplica un ranking según el costo de mantenimiento por kilómetro recorrido, de menor a mayor.

Problema de negocio que resuelve
Esta query permite evaluar qué tipo de vehículo resulta más conveniente desde el punto de vista operativo y económico. Es útil para comparar categorías de la flota, identificar cuáles son más costosas de sostener y apoyar decisiones sobre uso, renovación o mantenimiento.

Resultado esperado
El resultado muestra un resumen por tipo de vehículo con cantidad de unidades, viajes realizados, kilómetros recorridos, costo total de mantenimiento y costo por kilómetro, junto con un ranking de eficiencia de costo.




### Criterio de selección de índices

Para intentar mejorar el rendimiento de las consultas, se crearon cinco índices sobre columnas que aparecen repetidamente en los JOIN y en algunos filtros. La idea fue elegir campos que se usan mucho para relacionar tablas o para acotar resultados, ya que son los lugares donde un índice suele aportar más.

Los índices creados fueron los siguientes:´

CREATE INDEX idx_trips_status_departure
ON trips(status, departure_datetime);  ---> varias consultas trabajan con viajes completados y también usan la fecha de salida.

CREATE INDEX idx_trips_route_status
ON trips(route_id, status); ----> Muchas consultas relacionan los viajes con las rutas y además filtran por estado.

CREATE INDEX idx_trips_driver_departure
ON trips(driver_id, departure_datetime); ---> algunas consultas se analiza la actividad de los conductores y también se usa la fecha.

CREATE INDEX idx_deliveries_trip_id
ON deliveries(trip_id);  ---> porque deliveries se une muchas veces con trips por trip_id.

CREATE INDEX idx_maintenance_vehicle_id
ON maintenance(vehicle_id);  --->  la tabla de mantenimiento se relaciona con vehículos a través de vehicle_id.



### OPTMIZACION Y ANALISIS  DE TIMPO DE EJECUCION CON EXPLAIN ANALYZE

No se observó una mejora clara en los tiempos de ejecución después de crear los índices. Consultando con fuentes de internet y con ayuda de IA, entendí que esto puede ocurrir porque la mayoría de las queries del trabajo son consultas analíticas y no búsquedas puntuales. Es decir, no están pensadas para encontrar uno o pocos registros específicos, sino para recorrer muchas filas y realizar operaciones como conteos, promedios, sumas, agrupamientos y rankings. En estos casos, PostgreSQL muchas veces elige hacer un recorrido completo de la tabla (Seq Scan) en lugar de usar un índice, ya que igualmente necesita leer una gran parte de los datos para construir el resultado. Por eso, aunque los índices fueron creados sobre columnas importantes para joins y filtros frecuentes, su impacto en el tiempo de ejecución no siempre fue significativo.